In [ ]:
!pip install transformers datasets seqeval pandas huggingface_hub -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import json
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForTokenClassification,
    TrainingArguments, Trainer, DataCollatorForTokenClassification,
    pipeline
)
import numpy as np
from seqeval.metrics import classification_report, accuracy_score

In [ ]:
import pandas as pd
from datasets import Dataset

def load_conll_csv(file_path):
    df = pd.read_csv(file_path)
    df.fillna('', inplace=True)  # Ensure empty lines are handled
    sentences, ner_tags = [], []
    current_sentence, current_tags = [], []

    for _, row in df.iterrows():
        token, tag = row['token'], row['ner_tag']
        if token.strip() == '':
            if current_sentence:
                sentences.append(current_sentence)
                ner_tags.append([int(t) for t in current_tags])
                current_sentence, current_tags = [], []
        else:
            current_sentence.append(token)
            current_tags.append(tag)

    if current_sentence:  # Catch last sentence
        sentences.append(current_sentence)
        ner_tags.append([int(t) for t in current_tags])

    return Dataset.from_dict({'tokens': sentences, 'ner_tags': ner_tags})

train_dataset = load_conll_csv("train.csv")
valid_dataset = load_conll_csv("validation.csv")
test_dataset  = load_conll_csv("test.csv")

/tmp/ipython-input-1973971334.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.fillna('', inplace=True)  # Ensure empty lines are handled
/tmp/ipython-input-1973971334.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.fillna('', inplace=True)  # Ensure empty lines are handled
/tmp/ipython-input-1973971334.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.fillna('', inplace=True)  # Ensure empty lines are handled


In [ ]:
# Define id2label based on wikiANN-ta label scheme
id2label = {
    0: "O",         # Outside named entity
    1: "B-PER", 2: "I-PER",
    3: "B-ORG", 4: "I-ORG",
    5: "B-LOC", 6: "I-LOC"
}
label2id = {v: k for k, v in id2label.items()}

In [ ]:
from transformers import AutoTokenizer

model_checkpoint = "ai4bharat/indicNER"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_and_align_labels(example):
    tokenized_inputs = tokenizer(example["tokens"], truncation=True, is_split_into_words=True)
    labels = []
    word_ids = tokenized_inputs.word_ids()
    prev_word_id = None

    for word_id in word_ids:
        if word_id is None:
            labels.append(-100)
        elif word_id != prev_word_id:
            labels.append(example["ner_tags"][word_id])
        else:
            labels.append(-100)
        prev_word_id = word_id

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_train = train_dataset.map(tokenize_and_align_labels)
tokenized_valid = valid_dataset.map(tokenize_and_align_labels)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/346 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/15000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification
import numpy as np
from seqeval.metrics import classification_report, accuracy_score

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(id2label),
    id2label=id2label,
    label2id=label2id
)

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/667M [00:00<?, ?B/s]

In [ ]:
args = TrainingArguments(
    output_dir="/content/drive/MyDrive/NER_Models/indicNER_ta_v2",  # ✅ Drive path
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,              # 🔼 Increased learning rate slightly
    per_device_train_batch_size=16,  # 🔼 Increased batch size (if memory allows)
    per_device_eval_batch_size=16,
    num_train_epochs=5,              # 🔼 More epochs to improve generalization
    weight_decay=0.01,
    logging_dir="/content/drive/MyDrive/NER_Models/logs_v2",
    logging_steps=50,                # ✅ More frequent logging
    save_total_limit=2,              # ✅ Keep only latest 2 checkpoints
    load_best_model_at_end=True,     # ✅ Restore best checkpoint
    metric_for_best_model="f1",      # ✅ Choose a suitable metric
    greater_is_better=True,
    fp16=True,                       # ✅ Enable mixed precision if using GPU (Colab typically has T4/V100)
    report_to="none"
)

data_collator = DataCollatorForTokenClassification(tokenizer)

In [ ]:
def compute_metrics(p):
    predictions = np.argmax(p.predictions, axis=2)
    labels = p.label_ids

    true_preds = [
        [id2label[p] for (p, l) in zip(pred, label) if l != -100]
        for pred, label in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for (p, l) in zip(pred, label) if l != -100]
        for pred, label in zip(predictions, labels)
    ]

    return {
        "accuracy": accuracy_score(true_labels, true_preds),
        "f1": classification_report(true_labels, true_preds, output_dict=True)["weighted avg"]["f1-score"]
    }

In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

/tmp/ipython-input-19239566.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()

/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.016600,0.341291,0.950473,0.861248
2,0.017000,0.307980,0.953766,0.866737
3,0.005100,0.330910,0.955961,0.872786
4,0.003700,0.337673,0.955001,0.873268
5,0.008500,0.350679,0.956098,0.871960


/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


TrainOutput(global_step=4690, training_loss=0.009030335686449557, metrics={'train_runtime': 689.6979, 'train_samples_per_second': 108.743, 'train_steps_per_second': 6.8, 'total_flos': 1981204820425776.0, 'train_loss': 0.009030335686449557, 'epoch': 5.0})

In [ ]:
trainer.evaluate()

/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


{'eval_loss': 0.33767253160476685,
 'eval_accuracy': 0.955000685965153,
 'eval_f1': 0.8732681193153905,
 'eval_runtime': 1.9136,
 'eval_samples_per_second': 522.573,
 'eval_steps_per_second': 32.922,
 'epoch': 5.0}

In [ ]:
save_path = "/content/drive/MyDrive/NER_Models/indicNER_ta_v2"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

('/content/drive/MyDrive/NER_Models/indicNER_ta_v2/tokenizer_config.json',
 '/content/drive/MyDrive/NER_Models/indicNER_ta_v2/special_tokens_map.json',
 '/content/drive/MyDrive/NER_Models/indicNER_ta_v2/vocab.txt',
 '/content/drive/MyDrive/NER_Models/indicNER_ta_v2/added_tokens.json',
 '/content/drive/MyDrive/NER_Models/indicNER_ta_v2/tokenizer.json')

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

# Load model and tokenizer
model_path = "/content/drive/MyDrive/NER_Models/indicNER_ta_v2"
model = AutoModelForTokenClassification.from_pretrained(model_path, local_files_only=True)
tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=True, local_files_only=True)  # Must use fast tokenizer

def predict_ner_wordwise(text):
    """
    Predicts NER tags for each word using BIO scheme.
    """
    # Tokenize input
    encoding = tokenizer(text, return_offsets_mapping=True, return_tensors="pt", truncation=True)

    offset_mapping = encoding.pop("offset_mapping")  # Remove before model input

    with torch.no_grad():
        outputs = model(**encoding)

    predictions = torch.argmax(outputs.logits, dim=2)[0]
    tokens = tokenizer.convert_ids_to_tokens(encoding['input_ids'][0])
    word_ids = encoding.word_ids()

    word_to_label = {}
    for idx, word_id in enumerate(word_ids):
        if word_id is None:
            continue
        token = tokens[idx]
        label = model.config.id2label[predictions[idx].item()]

        if word_id not in word_to_label:
            word_to_label[word_id] = {"tokens": [token], "labels": [label]}
        else:
            word_to_label[word_id]["tokens"].append(token)
            word_to_label[word_id]["labels"].append(label)

    # Merge tokens back into words
    print("\nNER Tags:")
    for word in word_to_label.values():
        word_text = tokenizer.convert_tokens_to_string(word["tokens"]).replace(" ", "")
        label_sequence = word["labels"]
        label = label_sequence[0]  # Use first label (B- or O)

        # Adjust if all are "O"
        if all(l == "O" for l in label_sequence):
            label = "O"
        print(f"{word_text} : {label}")

In [ ]:
# Example usage
predict_ner_wordwise("மு.க.ஸ்டாலின் சென்னையில் பேசினார்")


NER Tags:
மு : B-PER
. : I-PER
க : I-PER
. : I-PER
ஸடாலின : I-PER
செனனையில : B-LOC
பேசினார : O


In [ ]:
predict_ner_wordwise("இந்திய கிரிக்கெட் கேப்டன் சச்சின் டெண்டுல்கர் சிறப்பாக விளையாடினார்")


NER Tags:
இநதிய : B-LOC
கிரிககெட : I-LOC
கேபடன : O
சசசின : B-PER
டெணடுலகர : I-PER
சிறபபாக : O
விளையாடினார : O


In [ ]:
predict_ner_wordwise("இந்திய கேப்டன் விராட் கோலி சிறப்பாக விளையாடினார்")


NER Tags:
இநதிய : B-LOC
கேபடன : O
விராட : B-PER
கோலி : I-PER
சிறபபாக : O
விளையாடினார : O


In [ ]:
predict_ner_wordwise("ஸ்டார்டப் இந்தியா நிகழ்ச்சியில் நரேந்திர மோடி பேசினார்")


NER Tags:
ஸடாரடப : B-ORG
இநதியா : I-ORG
நிகழசசியில : O
நரேநதிர : B-PER
மோடி : I-PER
பேசினார : O


In [ ]:
predict_ner_wordwise("சென்னை சூப்பர் கிங்ஸ் ஐபிஎல் போட்டியில் வெற்றி பெற்றது")


NER Tags:
செனனை : B-ORG
சூபபர : I-ORG
கிஙஸ : I-ORG
ஐபிஎல : O
போடடியில : O
வெறறி : O
பெறறது : O


In [ ]:
predict_ner_wordwise("கேப் டவுன் நகரம் தென் ஆப்ரிக்காவில் அமைந்துள்ளது")


NER Tags:
கேப : B-LOC
டவுன : I-LOC
நகரம : I-LOC
தென : B-LOC
ஆபரிககாவில : I-LOC
அமைநதுளளது : O


In [ ]:
predict_ner_wordwise("சுந்தர் பிச்சை கூகுள் நிறுவனத்தின் தலைவராக இருக்கிறார்")


NER Tags:
சுநதர : B-PER
பிசசை : I-PER
கூகுள : B-ORG
நிறுவனததின : O
தலைவராக : O
இருககிறார : O


In [ ]:
predict_ner_wordwise("ஜவாஹர்லால் நேரு இந்தியாவின் முதல் பிரதமராக இருந்தார்")


NER Tags:
ஜவாஹரலால : B-PER
நேரு : I-PER
இநதியாவின : B-LOC
முதல : O
பிரதமராக : O
இருநதார : O


In [ ]:
predict_ner_wordwise("சச்சின் டெண்டுல்கர் மும்பையில் பிறந்தார்")


NER Tags:
சசசின : O
டெணடுலகர : O
முமபையில : B-LOC
பிறநதார : O


In [ ]:
predict_ner_wordwise("விராட் கோலி தில்லியில் பிறந்தார்")


NER Tags:
விராட : B-PER
கோலி : I-PER
திலலியில : B-LOC
பிறநதார : O



NER Tags:
மதுரை : B-ORG
மாநகராடசி : I-ORG
சொதது : O
வரிவிதிபபு : O
முறைகேடு : O
குறிதது : O
சி : O
. : O
பி : O
. : O
ஐ : O
. : O
விசாரணை : O
கோரி : O
வழககு : O
வழககு : O
தொடரபபடடு : O
உளளது : O
. : O
ஏறகனவே : O
உயர : B-ORG
நதிமனற : I-ORG
உததரவு : O
அடிபபடையில : O
மதுரை : B-ORG
போலஸ : I-ORG
டி : O
. : I-ORG
ஐ : I-ORG
. : I-ORG
ஜி : I-ORG
. : O
தலைமையிலான : O
சிறபபு : O
குழு : O
வரிவிதிபபு : O
முறைகேடுவை : O
விசாரிதது : O
வருகிறது : O
. : O


In [ ]:
predict_ner_wordwise("தில்லியில் பிறந்த விராட் கோலி சிறப்பாக விளையாடினார். அவர் சென்னையில், இந்திய கிரிக்கெட் வாரியத்தின் ஏற்பாட்டில் நடைபெற்ற போட்டியில் அசத்தினார்.")


NER Tags:
திலலியில : B-LOC
பிறநத : O
விராட : B-PER
கோலி : I-PER
சிறபபாக : O
விளையாடினார : O
. : O
அவர : O
செனனையில : B-LOC
, : O
இநதிய : B-ORG
கிரிககெட : I-ORG
வாரியததின : I-ORG
ஏறபாடடில : O
நடைபெறற : O
போடடியில : O
அசததினார : O
. : O
